# Complex stft unet training

This notebook is part of the thesis notebook set.
It uses the shared prepared 70/30 speech/music split directory: `/content/drive/MyDrive/master/prepared_datasets/audio_70speech_30music_v1/splits`.

Purpose:
- This notebook trains a complex-STFT U-Net baseline.
- Checkpoint prefixes and manual checkpoint paths are configuration fields, not fixed thesis assumptions.
- The model-specific training or evaluation logic is kept from the original notebook unless the configuration depended on an old data split.


In [ ]:

from google.colab import drive
drive.mount('/content/drive', force_remount=True)

In [ ]:

!pip -q install soundfile librosa

In [ ]:

import os, json, time, random, math
from pathlib import Path

import numpy as np
import pandas as pd
import soundfile as sf
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
import torch.nn.functional as F
import torchaudio

from IPython.display import Audio, display

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("device:", device)
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

## Configuration

In [ ]:

SEED = 7
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED); torch.cuda.manual_seed_all(SEED)

DRIVE_ROOT = Path("/content/drive/MyDrive/master")
PREPARED_DATASET_DIR = DRIVE_ROOT / "prepared_datasets" / "audio_70speech_30music_v1"
SPLIT_DIR = PREPARED_DATASET_DIR / "splits"
assert DRIVE_ROOT.exists(), f"Missing DRIVE_ROOT: {DRIVE_ROOT}"

MODE = "A1CPLX"
RUN_NAME = "complex_stft_unet"
RUN_TAG = f"{RUN_NAME}_{time.strftime('%Y%m%d_%H%M%S')}"
CKPT_DIR = DRIVE_ROOT / "checkpoints_runs" / RUN_TAG
CKPT_DIR.mkdir(parents=True, exist_ok=True)

# audio / dataset
SR = 22050
SEG_S = 2.0
BATCH = 4
P_SPEECH = 0.70
NUM_WORKERS = 2
PIN_MEMORY = True

# training
STEPS = 40000
SAVE_EVERY = 2500
LOG_EVERY = 100

# complex STFT setup
N_FFT = 1024
HOP = 256
WIN = 1024
CENTER = True
F_BINS = N_FFT // 2 + 1

# inpainting
K_CHOICES = (1, 2)

# model
G_BASE = 24
G_GROUPS = 8
G_IN_CH = 3   # real, imag, mask
G_OUT_CH = 2  # delta real, delta imag

# optimizer
LR_G = 2e-4
WEIGHT_DECAY = 1e-4

# losses
LAMBDA_COMPLEX = 1.0
LAMBDA_LOGMAG = 20.0
LAMBDA_PHASE = 5.0
LAMBDA_WAV_MRSTFT = 3.0
LAMBDA_DELTA = 0.02

HF_START_FRAC = 0.45
HF_END_GAIN = 6.0
HF_RAMP_POWER = 2.0

COMPLEX_MASK_DILATE = 1
PHASE_MASK_DILATE = 2
WAV_MRSTFT_FFTS = (512, 1024, 2048)
WAV_MRSTFT_HOPS = (128, 256, 512)
WAV_MRSTFT_WINS = (512, 1024, 2048)
WAV_MRSTFT_LOGMAG = True

RESUME_PATH = None  # set to a .pt path when continuing a run
RESUME_LR_SCALE = 0.5

RUN_CONFIG = dict(
    seed=SEED,
    mode=MODE,
    run_name=RUN_NAME,
    sr=SR,
    segment_s=SEG_S,
    batch=BATCH,
    p_speech=P_SPEECH,
    stft=dict(n_fft=N_FFT, hop=HOP, win=WIN, center=CENTER),
    k_choices=list(K_CHOICES),
    g_arch="ComplexSTFTUNet2D",
    g_base=G_BASE,
    g_groups=G_GROUPS,
    losses=dict(
        lambda_complex=LAMBDA_COMPLEX,
        lambda_logmag=LAMBDA_LOGMAG,
        lambda_phase=LAMBDA_PHASE,
        lambda_wav_mrstft=LAMBDA_WAV_MRSTFT,
        lambda_delta=LAMBDA_DELTA,
        hf_start_frac=HF_START_FRAC,
        hf_end_gain=HF_END_GAIN,
        hf_ramp_power=HF_RAMP_POWER,
        complex_mask_dilate=COMPLEX_MASK_DILATE,
        phase_mask_dilate=PHASE_MASK_DILATE,
    ),
    opt_g=dict(lr=LR_G, weight_decay=WEIGHT_DECAY),
)
(CKPT_DIR / "run_config.json").write_text(json.dumps(RUN_CONFIG, indent=2), encoding="utf-8")
print("CKPT_DIR:", CKPT_DIR)

## Stage audio locally for faster training

In [ ]:

import shutil

DRIVE_SPLIT_DIR = SPLIT_DIR
LOCAL_DATA_ROOT = Path("/content/audio_70speech_30music_v1_cache")
LOCAL_SPLIT_DIR = Path("/content/audio_70speech_30music_v1_splits_local")

LOCAL_DATA_ROOT.mkdir(parents=True, exist_ok=True)
assert SPLIT_DIR.exists(), f"Missing prepared split directory: {SPLIT_DIR}"

LISTS = ["speech_train.txt", "music_train.txt", "speech_val.txt", "music_val.txt", "speech_test.txt", "music_test.txt"]

def _read_lines(p: Path):
    lines = [ln.strip().strip('"').strip("'") for ln in p.read_text().splitlines()]
    return [ln for ln in lines if ln]

def to_local_path(p: Path) -> Path:
    p = Path(p)
    for root in [DRIVE_ROOT, Path("/content/drive")]:
        try:
            rel = p.relative_to(root)
            return LOCAL_DATA_ROOT / rel
        except Exception:
            pass
    return LOCAL_DATA_ROOT / "flat" / p.name

all_files = []
for name in LISTS:
    src_list = DRIVE_SPLIT_DIR / name
    assert src_list.exists(), f"Missing split list: {src_list}"
    all_files += [Path(x) for x in _read_lines(src_list)]

all_files = sorted(set(all_files))
print("Unique audio files referenced by splits:", len(all_files))

copied = 0
skipped = 0
for src in all_files:
    dst = to_local_path(src)
    dst.parent.mkdir(parents=True, exist_ok=True)
    if dst.exists():
        skipped += 1
        continue
    shutil.copy2(src, dst)
    copied += 1

print(f"Copied {copied} files (skipped {skipped} already present).")
print("Local cache root:", LOCAL_DATA_ROOT)

for name in LISTS:
    src_list = DRIVE_SPLIT_DIR / name
    src_paths = [Path(x) for x in _read_lines(src_list)]
    dst_paths = [str(to_local_path(p)) for p in src_paths]
    (LOCAL_SPLIT_DIR / name).write_text("\n".join(dst_paths) + "\n", encoding="utf-8")

SPLIT_DIR = LOCAL_SPLIT_DIR
RUN_CONFIG["split_dir"] = str(SPLIT_DIR)
RUN_CONFIG["local_data_root"] = str(LOCAL_DATA_ROOT)
(CKPT_DIR / "run_config.json").write_text(json.dumps(RUN_CONFIG, indent=2), encoding="utf-8")

for name in LISTS:
    p = SPLIT_DIR / name
    assert p.exists(), f"Missing local list: {p}"
    first = p.read_text().splitlines()[0].strip().strip('"').strip("'")
    print(name, "first:", first)
    assert first.startswith("/content/"), "List still points to Drive paths!"
    assert Path(first).exists(), "First local wav missing!"

print("Using local cached wav paths.")

## Dataset + loaders

In [ ]:

from torch.utils.data import Dataset, DataLoader

def read_list(p: Path):
    assert p.exists(), f"Missing list: {p}"
    lines = [ln.strip().strip('"').strip("'") for ln in p.read_text().splitlines()]
    return [Path(ln) for ln in lines if ln]

def safe_load_wav_mono(path: Path, target_sr: int) -> torch.Tensor:
    path = Path(path)
    try:
        wav, sr = torchaudio.load(str(path))
        wav = wav.mean(dim=0)
    except Exception:
        data, sr = sf.read(str(path), dtype="float32", always_2d=True)
        wav = torch.from_numpy(data.T).mean(dim=0)

    if sr != target_sr:
        wav = torchaudio.functional.resample(wav, sr, target_sr)

    wav = wav.to(torch.float32)
    wav = wav / (wav.abs().max() + 1e-8)
    wav = (0.95 * wav).clamp(-1.0, 1.0)
    return wav

class FileListDataset(Dataset):
    def __init__(self, list_path: Path, sr: int, segment_s: float):
        self.paths = read_list(list_path)
        self.sr = sr
        self.seg_len = int(sr * segment_s)
        assert len(self.paths) > 0, f"Empty list: {list_path}"

    def __len__(self):
        return len(self.paths)

    def __getitem__(self, idx):
        w = safe_load_wav_mono(self.paths[idx], self.sr)
        if w.numel() < self.seg_len:
            w = F.pad(w, (0, self.seg_len - w.numel()))
        max_start = w.numel() - self.seg_len
        start = random.randint(0, max_start) if max_start > 0 else 0
        return w[start:start+self.seg_len]

speech_train = SPLIT_DIR / "speech_train.txt"
music_train  = SPLIT_DIR / "music_train.txt"
speech_val   = SPLIT_DIR / "speech_val.txt"
music_val    = SPLIT_DIR / "music_val.txt"

ds_speech = FileListDataset(speech_train, SR, SEG_S)
ds_music  = FileListDataset(music_train,  SR, SEG_S)

dl_speech = DataLoader(
    ds_speech, batch_size=BATCH, shuffle=True,
    num_workers=NUM_WORKERS, drop_last=True, pin_memory=PIN_MEMORY,
    persistent_workers=(NUM_WORKERS > 0),
)
dl_music = DataLoader(
    ds_music, batch_size=BATCH, shuffle=True,
    num_workers=NUM_WORKERS, drop_last=True, pin_memory=PIN_MEMORY,
    persistent_workers=(NUM_WORKERS > 0),
)

def infinite(dl):
    while True:
        for b in dl:
            yield b

it_speech = infinite(dl_speech)
it_music = infinite(dl_music)

def next_mixed_batch():
    b = next(it_speech) if random.random() < P_SPEECH else next(it_music)
    return b.to(device)

print("speech files:", len(ds_speech), "music files:", len(ds_music), "P_SPEECH:", P_SPEECH)

## Complex STFT helpers, inpainting, and losses

In [ ]:

def build_freq_weights(n_bins: int, start_frac: float, end_gain: float, power: float, device=None):
    idx = torch.linspace(0.0, 1.0, steps=n_bins, device=device)
    ramp = ((idx - start_frac).clamp_min(0.0) / max(1e-6, 1.0 - start_frac)) ** power
    w = 1.0 + (end_gain - 1.0) * ramp
    return w.view(1, n_bins, 1)

STFT_FREQ_WEIGHTS = build_freq_weights(F_BINS, HF_START_FRAC, HF_END_GAIN, HF_RAMP_POWER, device=device)

def stft_complex(wav_bt: torch.Tensor) -> torch.Tensor:
    window = torch.hann_window(WIN, device=wav_bt.device)
    return torch.stft(
        wav_bt,
        n_fft=N_FFT,
        hop_length=HOP,
        win_length=WIN,
        window=window,
        center=CENTER,
        return_complex=True,
    )

def istft_complex(X_bft: torch.Tensor, length: int) -> torch.Tensor:
    window = torch.hann_window(WIN, device=X_bft.device)
    return torch.istft(
        X_bft,
        n_fft=N_FFT,
        hop_length=HOP,
        win_length=WIN,
        window=window,
        center=CENTER,
        length=length,
    )

def linear_time_inpaint_complex(X: torch.Tensor, k: int, offset: int):
    """
    X: (B, F, T) complex
    returns:
      X_interp: (B, F, T) complex
      mask:     (B, 1, 1, T) float
    """
    B, Freq, T = X.shape
    step = k + 1

    Xr = X.real.clone()
    Xi = X.imag.clone()
    mask = torch.zeros((B, 1, 1, T), device=X.device, dtype=torch.float32)

    if k == 0:
        return torch.complex(Xr, Xi), mask

    for left in range(offset, T - step, step):
        right = left + step
        for j in range(1, step):
            t = left + j
            alpha = j / step
            Xr[:, :, t] = (1.0 - alpha) * X.real[:, :, left] + alpha * X.real[:, :, right]
            Xi[:, :, t] = (1.0 - alpha) * X.imag[:, :, left] + alpha * X.imag[:, :, right]
            mask[:, :, :, t] = 1.0

    return torch.complex(Xr, Xi), mask

def make_inpainting_pair_complex(X_real: torch.Tensor, k_choices=(1, 2), randomize_offset=True):
    k = int(random.choice(list(k_choices)))
    step = k + 1
    offset = int(random.randrange(step)) if randomize_offset else 0
    X_interp, mask = linear_time_inpaint_complex(X_real, k=k, offset=offset)
    return {"real": X_real, "interp": X_interp, "mask": mask, "k": k, "offset": offset}

def pack_complex_input(X_interp: torch.Tensor, mask: torch.Tensor):
    xr = X_interp.real.unsqueeze(1)
    xi = X_interp.imag.unsqueeze(1)
    xm = mask.expand(-1, 1, X_interp.shape[1], -1)
    return torch.cat([xr, xi, xm], dim=1)

def dilate_mask_time(mask: torch.Tensor, radius: int):
    if radius <= 0:
        return mask
    return F.max_pool2d(mask, kernel_size=(1, 2 * radius + 1), stride=1, padding=(0, radius))

def _expand_mask(mask: torch.Tensor, X: torch.Tensor, radius: int = 0):
    m = dilate_mask_time(mask, radius)
    return m[:, 0].expand(-1, X.shape[1], -1)  # (B,F,T)

def masked_mean(x: torch.Tensor, mask_ft: torch.Tensor, freq_weights=None, eps: float = 1e-8):
    w = mask_ft
    if freq_weights is not None:
        w = w * freq_weights
    return (x * w).sum() / (w.sum() + eps)

def masked_complex_l1(X_hat: torch.Tensor, X_tgt: torch.Tensor, mask: torch.Tensor, freq_weights=None, mask_dilate: int = 0):
    m = _expand_mask(mask, X_hat, radius=mask_dilate)
    err = (X_hat.real - X_tgt.real).abs() + (X_hat.imag - X_tgt.imag).abs()
    return masked_mean(err, m, freq_weights=freq_weights)

def masked_logmag_l1(X_hat: torch.Tensor, X_tgt: torch.Tensor, mask: torch.Tensor, freq_weights=None, mask_dilate: int = 0, eps: float = 1e-7):
    m = _expand_mask(mask, X_hat, radius=mask_dilate)
    err = (X_hat.abs().clamp_min(eps).log() - X_tgt.abs().clamp_min(eps).log()).abs()
    return masked_mean(err, m, freq_weights=freq_weights)

def masked_phase_cos_loss(X_hat: torch.Tensor, X_tgt: torch.Tensor, mask: torch.Tensor, freq_weights=None, mask_dilate: int = 0, eps: float = 1e-7):
    m = _expand_mask(mask, X_hat, radius=mask_dilate)
    phase_diff = torch.angle(X_hat) - torch.angle(X_tgt)
    base = 1.0 - torch.cos(phase_diff)
    mag_gate = (X_tgt.abs() > eps).float()
    return masked_mean(base * mag_gate, m, freq_weights=freq_weights)

def _stft_mag(x: torch.Tensor, n_fft: int, hop: int, win: int, eps: float = 1e-7):
    window = torch.hann_window(win, device=x.device)
    X = torch.stft(
        x,
        n_fft=n_fft,
        hop_length=hop,
        win_length=win,
        window=window,
        center=True,
        return_complex=True,
    )
    return X.abs().clamp_min(eps)

def mrstft_loss(
    x_hat: torch.Tensor,
    x_tgt: torch.Tensor,
    fft_sizes=(512, 1024, 2048),
    hop_sizes=(128, 256, 512),
    win_sizes=(512, 1024, 2048),
    use_logmag=True,
):
    assert len(fft_sizes) == len(hop_sizes) == len(win_sizes)
    total = 0.0
    n = 0
    for n_fft, hop, win in zip(fft_sizes, hop_sizes, win_sizes):
        mag_hat = _stft_mag(x_hat, n_fft=n_fft, hop=hop, win=win)
        mag_tgt = _stft_mag(x_tgt, n_fft=n_fft, hop=hop, win=win)
        l_mag = (mag_hat - mag_tgt).abs().mean()
        if use_logmag:
            l_log = (mag_hat.log() - mag_tgt.log()).abs().mean()
            total = total + l_mag + l_log
            n += 2
        else:
            total = total + l_mag
            n += 1
    return total / max(1, n)

## Generator - residual 2D U-Net over real/imag + mask

In [ ]:

def _valid_groups(ch: int, requested: int):
    for g in [requested, 8, 4, 2, 1]:
        if ch % g == 0:
            return g
    return 1

class ConvGNAct(nn.Module):
    def __init__(self, in_ch, out_ch, groups=8):
        super().__init__()
        g = _valid_groups(out_ch, groups)
        self.block = nn.Sequential(
            nn.Conv2d(in_ch, out_ch, kernel_size=3, padding=1),
            nn.GroupNorm(g, out_ch),
            nn.SiLU(inplace=True),
            nn.Conv2d(out_ch, out_ch, kernel_size=3, padding=1),
            nn.GroupNorm(g, out_ch),
            nn.SiLU(inplace=True),
        )

    def forward(self, x):
        return self.block(x)

class ComplexSTFTUNet2D(nn.Module):
    def __init__(self, in_ch=3, out_ch=2, base=24, groups=8):
        super().__init__()
        self.enc1 = ConvGNAct(in_ch, base, groups)
        self.enc2 = ConvGNAct(base, base * 2, groups)
        self.enc3 = ConvGNAct(base * 2, base * 4, groups)
        self.bot  = ConvGNAct(base * 4, base * 8, groups)

        self.dec3 = ConvGNAct(base * 8 + base * 4, base * 4, groups)
        self.dec2 = ConvGNAct(base * 4 + base * 2, base * 2, groups)
        self.dec1 = ConvGNAct(base * 2 + base, base, groups)

        self.out = nn.Conv2d(base, out_ch, kernel_size=3, padding=1)
        self.pool = nn.AvgPool2d(kernel_size=2)

    def forward(self, x):
        x1 = self.enc1(x)
        x2 = self.enc2(self.pool(x1))
        x3 = self.enc3(self.pool(x2))
        xb = self.bot(self.pool(x3))

        y3 = F.interpolate(xb, size=x3.shape[-2:], mode="bilinear", align_corners=False)
        y3 = self.dec3(torch.cat([y3, x3], dim=1))

        y2 = F.interpolate(y3, size=x2.shape[-2:], mode="bilinear", align_corners=False)
        y2 = self.dec2(torch.cat([y2, x2], dim=1))

        y1 = F.interpolate(y2, size=x1.shape[-2:], mode="bilinear", align_corners=False)
        y1 = self.dec1(torch.cat([y1, x1], dim=1))

        return self.out(y1)

G = ComplexSTFTUNet2D(
    in_ch=G_IN_CH,
    out_ch=G_OUT_CH,
    base=G_BASE,
    groups=G_GROUPS,
).to(device)

n_params = sum(p.numel() for p in G.parameters())
print("G params:", f"{n_params/1e6:.2f}M")

## Optimizer + checkpoint / resume

In [ ]:

opt_g = torch.optim.AdamW(G.parameters(), lr=LR_G, betas=(0.9, 0.99), weight_decay=WEIGHT_DECAY)

def save_ckpt(step: int):
    obj = {
        "step": step,
        "mode": MODE,
        "run_config": RUN_CONFIG,
        "G": G.state_dict(),
        "opt_g": opt_g.state_dict(),
        "stft_freq_weights": STFT_FREQ_WEIGHTS.detach().cpu(),
    }
    p = CKPT_DIR / f"{MODE}_step{step:07d}.pt"
    torch.save(obj, p)
    torch.save(obj, CKPT_DIR / "latest.pt")
    print("saved:", p)

start_step = 0
if RESUME_PATH is not None and Path(RESUME_PATH).exists():
    ck = torch.load(str(RESUME_PATH), map_location="cpu")
    G.load_state_dict(ck["G"], strict=True)
    opt_g.load_state_dict(ck["opt_g"])
    start_step = int(ck.get("step", 0))
    for pg in opt_g.param_groups:
        pg["lr"] = LR_G * RESUME_LR_SCALE
    print("Resumed:", RESUME_PATH)
    print("start_step:", start_step, "| lr scaled by", RESUME_LR_SCALE)
else:
    print("Starting fresh.")

## Training loop

In [ ]:

G.train()
t0 = time.time()

log_path = CKPT_DIR / "train_log.csv"
if not log_path.exists():
    log_path.write_text(
        "step,loss_total,loss_complex,loss_logmag,loss_phase,loss_wav_mrstft,loss_delta,"
        "base_logmag,ref_logmag,base_phase,ref_phase,base_complex,ref_complex,"
        "imp_logmag_pct,imp_phase_pct,imp_complex_pct,k,offset,minutes\n",
        encoding="utf-8",
    )

for step in range(start_step + 1, start_step + STEPS + 1):
    wav = next_mixed_batch()                                  # (B,T)
    X_real = stft_complex(wav)                                # (B,F,Tspec)
    pair = make_inpainting_pair_complex(X_real, k_choices=K_CHOICES, randomize_offset=True)
    X_interp, mask = pair["interp"], pair["mask"]

    with torch.no_grad():
        base_complex = masked_complex_l1(X_interp, X_real, mask, freq_weights=None, mask_dilate=COMPLEX_MASK_DILATE)
        base_logmag  = masked_logmag_l1(X_interp, X_real, mask, freq_weights=STFT_FREQ_WEIGHTS, mask_dilate=COMPLEX_MASK_DILATE)
        base_phase   = masked_phase_cos_loss(X_interp, X_real, mask, freq_weights=STFT_FREQ_WEIGHTS, mask_dilate=PHASE_MASK_DILATE)

    x_in = pack_complex_input(X_interp, mask)
    delta = G(x_in)                                           # (B,2,F,Tspec)
    dX = torch.complex(delta[:, 0], delta[:, 1])              # (B,F,Tspec)

    mask_ft = mask[:, 0].expand(-1, X_interp.shape[1], -1)    # (B,F,Tspec)
    X_hat = X_interp + dX * mask_ft

    wav_hat = istft_complex(X_hat, length=wav.shape[-1])

    loss_complex = masked_complex_l1(X_hat, X_real, mask, freq_weights=None, mask_dilate=COMPLEX_MASK_DILATE)
    loss_logmag  = masked_logmag_l1(X_hat, X_real, mask, freq_weights=STFT_FREQ_WEIGHTS, mask_dilate=COMPLEX_MASK_DILATE)
    loss_phase   = masked_phase_cos_loss(X_hat, X_real, mask, freq_weights=STFT_FREQ_WEIGHTS, mask_dilate=PHASE_MASK_DILATE)
    loss_wav_mrstft = mrstft_loss(
        wav_hat, wav,
        fft_sizes=WAV_MRSTFT_FFTS,
        hop_sizes=WAV_MRSTFT_HOPS,
        win_sizes=WAV_MRSTFT_WINS,
        use_logmag=WAV_MRSTFT_LOGMAG,
    )
    loss_delta = (dX.abs() * mask_ft).sum() / (mask_ft.sum() + 1e-8)

    loss_total = (
        LAMBDA_COMPLEX * loss_complex
        + LAMBDA_LOGMAG * loss_logmag
        + LAMBDA_PHASE * loss_phase
        + LAMBDA_WAV_MRSTFT * loss_wav_mrstft
        + LAMBDA_DELTA * loss_delta
    )

    opt_g.zero_grad(set_to_none=True)
    loss_total.backward()
    torch.nn.utils.clip_grad_norm_(G.parameters(), 5.0)
    opt_g.step()

    with torch.no_grad():
        ref_complex = masked_complex_l1(X_hat, X_real, mask, freq_weights=None, mask_dilate=COMPLEX_MASK_DILATE)
        ref_logmag  = masked_logmag_l1(X_hat, X_real, mask, freq_weights=STFT_FREQ_WEIGHTS, mask_dilate=COMPLEX_MASK_DILATE)
        ref_phase   = masked_phase_cos_loss(X_hat, X_real, mask, freq_weights=STFT_FREQ_WEIGHTS, mask_dilate=PHASE_MASK_DILATE)

        imp_logmag_pct = 100.0 * (base_logmag.item() - ref_logmag.item()) / max(1e-8, base_logmag.item())
        imp_phase_pct  = 100.0 * (base_phase.item()  - ref_phase.item())  / max(1e-8, base_phase.item())
        imp_complex_pct= 100.0 * (base_complex.item()- ref_complex.item())/ max(1e-8, base_complex.item())

    if (step % LOG_EVERY == 0) or (step == start_step + 1):
        mins = (time.time() - t0) / 60.0
        row = [
            step,
            float(loss_total.item()),
            float(loss_complex.item()),
            float(loss_logmag.item()),
            float(loss_phase.item()),
            float(loss_wav_mrstft.item()),
            float(loss_delta.item()),
            float(base_logmag.item()),
            float(ref_logmag.item()),
            float(base_phase.item()),
            float(ref_phase.item()),
            float(base_complex.item()),
            float(ref_complex.item()),
            float(imp_logmag_pct),
            float(imp_phase_pct),
            float(imp_complex_pct),
            int(pair["k"]),
            int(pair["offset"]),
            float(mins),
        ]
        with log_path.open("a", encoding="utf-8") as f:
            f.write(",".join(map(str, row)) + "\n")

        print(
            f"step {step:7d} | "
            f"loss {loss_total.item():.4f} | "
            f"complex {loss_complex.item():.4f} | "
            f"logmag {loss_logmag.item():.4f} | "
            f"phase {loss_phase.item():.4f} | "
            f"wav {loss_wav_mrstft.item():.4f} | "
            f"imp_logmag {imp_logmag_pct:6.2f}% | "
            f"imp_phase {imp_phase_pct:6.2f}% | "
            f"k={pair['k']} off={pair['offset']}"
        )

    if step % SAVE_EVERY == 0:
        save_ckpt(step)

save_ckpt(start_step + STEPS)
print("done")

## Quick sanity plots from the training log

In [ ]:

log_df = pd.read_csv(log_path)
display(log_df.tail())

plt.figure(figsize=(9,4))
plt.plot(log_df["step"], log_df["loss_total"], label="loss_total")
plt.plot(log_df["step"], log_df["loss_logmag"], label="loss_logmag")
plt.plot(log_df["step"], log_df["loss_phase"], label="loss_phase")
plt.plot(log_df["step"], log_df["loss_wav_mrstft"], label="loss_wav_mrstft")
plt.legend()
plt.grid(True, alpha=0.3)
plt.title(RUN_TAG + ": losses")
plt.show()

plt.figure(figsize=(9,4))
plt.plot(log_df["step"], log_df["imp_logmag_pct"], label="imp_logmag_pct")
plt.plot(log_df["step"], log_df["imp_phase_pct"], label="imp_phase_pct")
plt.plot(log_df["step"], log_df["imp_complex_pct"], label="imp_complex_pct")
plt.axhline(0.0, ls="--", c="k", lw=1)
plt.legend()
plt.grid(True, alpha=0.3)
plt.title(RUN_TAG + ": improvement over interpolation")
plt.show()

## Quick eval on one file (direct ISTFT listening)

In [ ]:

@torch.no_grad()
def eval_one_file(path: Path, k=1, offset=0, do_listen=False):
    G.eval()
    w = safe_load_wav_mono(path, SR)[: int(SR * SEG_S)].to(device)
    wb = w.unsqueeze(0)

    X_real = stft_complex(wb)
    X_interp, mask = linear_time_inpaint_complex(X_real, k=k, offset=offset)

    x_in = pack_complex_input(X_interp, mask)
    delta = G(x_in)
    dX = torch.complex(delta[:, 0], delta[:, 1])
    mask_ft = mask[:, 0].expand(-1, X_interp.shape[1], -1)
    X_hat = X_interp + dX * mask_ft

    wav_interp = istft_complex(X_interp, length=w.shape[-1])
    wav_hat = istft_complex(X_hat, length=w.shape[-1])

    base_complex = masked_complex_l1(X_interp, X_real, mask, mask_dilate=COMPLEX_MASK_DILATE).item()
    ref_complex  = masked_complex_l1(X_hat,   X_real, mask, mask_dilate=COMPLEX_MASK_DILATE).item()
    base_logmag  = masked_logmag_l1(X_interp, X_real, mask, freq_weights=STFT_FREQ_WEIGHTS, mask_dilate=COMPLEX_MASK_DILATE).item()
    ref_logmag   = masked_logmag_l1(X_hat,    X_real, mask, freq_weights=STFT_FREQ_WEIGHTS, mask_dilate=COMPLEX_MASK_DILATE).item()
    base_phase   = masked_phase_cos_loss(X_interp, X_real, mask, freq_weights=STFT_FREQ_WEIGHTS, mask_dilate=PHASE_MASK_DILATE).item()
    ref_phase    = masked_phase_cos_loss(X_hat,    X_real, mask, freq_weights=STFT_FREQ_WEIGHTS, mask_dilate=PHASE_MASK_DILATE).item()

    print(
        f"{path.name} | k={k} off={offset} | "
        f"complex {base_complex:.6f}->{ref_complex:.6f} | "
        f"logmag {base_logmag:.6f}->{ref_logmag:.6f} | "
        f"phase {base_phase:.6f}->{ref_phase:.6f}"
    )

    if do_listen:
        print("Original")
        display(Audio(w.detach().cpu().numpy(), rate=SR))
        print("Interpolated complex STFT")
        display(Audio(wav_interp[0].detach().cpu().numpy(), rate=SR))
        print("Fixed complex STFT")
        display(Audio(wav_hat[0].detach().cpu().numpy(), rate=SR))

    G.train()
    return {
        "wav_real": w.detach().cpu(),
        "wav_interp": wav_interp[0].detach().cpu(),
        "wav_hat": wav_hat[0].detach().cpu(),
        "X_real": X_real[0].detach().cpu(),
        "X_interp": X_interp[0].detach().cpu(),
        "X_hat": X_hat[0].detach().cpu(),
    }

speech_path = read_list(speech_val)[0]
music_path = read_list(music_val)[0]

print("Speech sanity check:")
_ = eval_one_file(speech_path, k=1, offset=0, do_listen=False)

print("\nMusic sanity check:")
_ = eval_one_file(music_path, k=1, offset=0, do_listen=False)

## Notes 
This notebook is the representation-change experiment.
If this helps, the next comparison notebook should compare:
- interpolation
- current best A1 2D mel U-Net
- this complex STFT U-Net

Because this run outputs direct waveform via ISTFT, it needs its own eval notebook, not the current BigVGAN mel-only one.